In [1]:
import pandas as pd
import lxml
import requests
import yfinance as yf
from io import StringIO

In [2]:
url= "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers= {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [3]:
response=requests.get(url, headers=headers)

sp500=pd.read_html(StringIO(response.text))[0]
sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [4]:
sp500['Symbol']=sp500['Symbol'].str.replace(".", "-", regex=False)

In [5]:
# list of ticker symbols for download (Needs to be done for S&P 500)
tickerStrings = sp500['Symbol'].tolist()

In [24]:
df = yf.download(tickerStrings, group_by='Ticker', period='1d')

[*******               14%                       ]  71 of 503 completed$MGM: possibly delisted; no price data found  (period=1d)
[*********             19%                       ]  97 of 503 completed$ALB: possibly delisted; no price data found  (period=1d)
[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['MGM', 'ALB']: possibly delisted; no price data found  (period=1d)


In [25]:
df

Ticker             GM                                              MDT  \
Price            Open    High        Low   Close     Volume       Open   
Date                                                                     
2026-06-23        NaN     NaN        NaN     NaN        NaN        NaN   
2026-06-24  79.129997  80.569  79.080002  79.625  4810437.0  80.120003   

Ticker                                                  ...      O             \
Price            High        Low      Close     Volume  ...   Open       High   
Date                                                    ...                     
2026-06-23        NaN        NaN        NaN        NaN  ...    NaN        NaN   
2026-06-24  82.089996  80.120003  80.599998  4450147.0  ...  61.77  62.455002   

Ticker                                         HOOD                         \
Price         Low      Close     Volume        Open        High        Low   
Date                                                                         
2026-06-23    NaN        NaN        NaN         NaN         NaN        NaN   
2026-06-24  61.48  62.169998  4059659.0  102.346001  104.269997  96.294998   

Ticker                          
Price        Close      Volume  
Date                            
2026-06-23     NaN         NaN  
2026-06-24  97.195  24692367.0  

[2 rows x 2517 columns]

In [26]:
df = df.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
df

Price,Date,Ticker,Open,High,Low,Close,Volume,Adj Close
0,2026-06-23,GM,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-23,MDT,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-23,NDSN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-23,UNP,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-23,UBER,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1001,2026-06-24,ULTA,465.899994,481.739990,459.000000,474.605011,690981.0,NaN
1002,2026-06-24,ZBRA,238.619995,249.419998,237.000000,246.660004,548355.0,NaN
1003,2026-06-24,LITE,824.744995,866.429993,803.330017,835.440002,3431691.0,NaN
1004,2026-06-24,O,61.770000,62.455002,61.480000,62.169998,4059659.0,NaN


In [27]:
downloaded_tickers=set(df["Ticker"].unique())

In [28]:
missing = set(tickerStrings) - downloaded_tickers

missing

set()

In [29]:
len(downloaded_tickers)

503

In [30]:
row_counts = df.groupby("Ticker").size()
print(row_counts.sort_values())

Ticker
A       2
AAPL    2
ABBV    2
ABNB    2
ABT     2
       ..
XYZ     2
YUM     2
ZBH     2
ZBRA    2
ZTS     2
Length: 503, dtype: int64


In [31]:
for ticker in ["FRT", "PHM", "SYY"]:
    count = (df["Ticker"] == ticker).sum()
    print(f"{ticker}: {count} rows")

FRT: 2 rows
PHM: 2 rows
SYY: 2 rows


In [32]:
df.columns.name = None
df

,Date,Ticker,Open,High,Low,Close,Volume,Adj Close
0,2026-06-23,GM,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-23,MDT,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-23,NDSN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-23,UNP,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-23,UBER,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1001,2026-06-24,ULTA,465.899994,481.739990,459.000000,474.605011,690981.0,NaN
1002,2026-06-24,ZBRA,238.619995,249.419998,237.000000,246.660004,548355.0,NaN
1003,2026-06-24,LITE,824.744995,866.429993,803.330017,835.440002,3431691.0,NaN
1004,2026-06-24,O,61.770000,62.455002,61.480000,62.169998,4059659.0,NaN


In [33]:
def validate_download(df, expected_tickers, ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]):
    
    downloaded_tickers = set(df["Ticker"].unique())
    fully_missing = set(expected_tickers) - downloaded_tickers

    # Per-column null counts per ticker
    null_by_col = (
        df.groupby("Ticker")[ohlcv_cols]
        .apply(lambda x: x.isna().sum())
    )
    null_by_col["Total"] = null_by_col.sum(axis=1)
    has_nulls = null_by_col[null_by_col["Total"] > 0]

    # First and last valid (non-null) date per ticker
    valid_dates = (
        df[df["Close"].notna()]
        .groupby("Ticker")["Date"]
        .agg(
            first_valid_date="min",
            last_valid_date="max"
        )
    )

    # Get the earliest date in your entire dataset (start of your backfill window)
    backfill_start = df["Date"].min()
    
    # Count trading days between backfill start and first valid date per ticker
    # Trading days = number of dates that appear in your df for any ticker
    all_trading_days = df["Date"].drop_duplicates().sort_values()
    
    def count_trading_days_before(first_valid_date):
        return (all_trading_days < first_valid_date).sum()
    
    valid_dates["trading_days_before_listing"] = valid_dates["first_valid_date"].apply(
        count_trading_days_before
    )
    # Distinguish fully null vs partially null
    row_counts = df.groupby("Ticker").size()
    max_possible_nulls = row_counts * len(ohlcv_cols)
    fully_null_mask = has_nulls["Total"] == max_possible_nulls[has_nulls.index]
    fully_null = has_nulls[fully_null_mask]
    partially_null = has_nulls[~fully_null_mask].sort_values("Total", ascending=False)

    # Join valid date info onto partially null summary
    partially_null_summary = partially_null.join(valid_dates, how="left")

    # Report
    print(f"Total tickers expected:  {len(expected_tickers)}")
    print(f"Total tickers downloaded: {len(downloaded_tickers)}")

    if fully_missing:
        print(f"\nFully missing (not in df at all): {fully_missing}")
    else:
        print("\nFully missing: none")

    if len(fully_null) > 0:
        print(f"\nPresent but completely null (failed download):")
        print(fully_null.to_string())
    else:
        print("\nCompletely null tickers: none")

    if len(partially_null_summary) > 0:
        print(f"\nPartially null (recent IPO/spin-off) — breakdown by column + valid date range:\n")
        print(partially_null_summary.to_string())
    else:
        print("\nPartially null tickers: none")

    return fully_missing, fully_null, partially_null_summary

In [34]:
fully_missing, fully_null, partially_null = validate_download(df, tickerStrings)

Total tickers expected:  503
Total tickers downloaded: 503

Fully missing: none

Present but completely null (failed download):
        Open  High  Low  Close  Volume  Total
Ticker                                       
ALB        2     2    2      2       2     10
MGM        2     2    2      2       2     10

Partially null (recent IPO/spin-off) — breakdown by column + valid date range:

        Open  High  Low  Close  Volume  Total first_valid_date last_valid_date  trading_days_before_listing
Ticker                                                                                                     
A          1     1    1      1       1      5       2026-06-24      2026-06-24                            1
AAPL       1     1    1      1       1      5       2026-06-24      2026-06-24                            1
ABBV       1     1    1      1       1      5       2026-06-24      2026-06-24                            1
ABNB       1     1    1      1       1      5       2026-06-24     

In [35]:
df.to_csv('daily_data_2306.csv')

In [36]:
ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]

In [37]:
null_counts_by_ticker = (
    df.groupby("Ticker")[ohlcv_cols]
    .apply(lambda x: x.isna().sum().sum())  # total nulls across all OHLCV cols, per ticker
    .sort_values(ascending=False)
)

print(null_counts_by_ticker.head(10))

Ticker
ALB     10
MGM     10
NFLX     5
NI       5
NKE      5
NOC      5
NOW      5
NRG      5
NEM      5
NTAP     5
dtype: int64


In [38]:
yf.download("GOOG", group_by='Ticker', period='1d')

[*********************100%***********************]  1 of 1 completed


Ticker            GOOG                                             
Price             Open        High        Low       Close    Volume
Date                                                               
2026-06-24  348.720001  352.829987  341.51001  343.339996  16445920